# TRACER Segmented Workflow

This notebook walks through the production TRACER workflow on a dataset that already carries a per-transcript `cell_id` column from upstream segmentation (e.g. Xenium nucleus-based segmentation).

The pipeline takes the input segmentation as a starting point and refines it using transcript-level gene-coherence and spatial structure. Output is a new per-transcript label that resolves Mickey-Mouse-shape cells, gene-incoherent assignments, and unassigned transcripts that belong to coherent neighborhoods.

## Stages

| step | name | what it does |
|---|---|---|
| 0 | Bootstrap NPMI | compute a panel-wide gene-pair PMI matrix (one-time) |
| 1 | **Prune** | drop gene-incoherent transcripts from each input cell |
| 2 | **Split** | break spatially disconnected cells into fragments |
| 3 | **Initial Rescue** | absorb unassigned transcripts into compatible existing entities |
| 4 | **Group** | cluster the remaining unassigned transcripts into new components |
| 5 | **Stitch** | merge gene-compatible neighboring entities |
| 6 | **Demote** | drop entities below the size threshold |
| 7 | **Final Rescue** | absorb residual unassigned transcripts into surviving entities |

After each step the notebook prints the current `cells / partials / components / unassigned tx` breakdown, and a final cumulative table at the bottom shows how each transformation moves transcripts through the categories.

## Setup

In [ ]:
from __future__ import annotations

import math
import time
from pathlib import Path

import numpy as np
import pandas as pd

from tracer.data import discover_data_files, load_full_df
from tracer.metrics import compute_npmi_bootstrap
from tracer.pruning import prune_transcripts_fast, prune_genes_by_npmi_greedy
from tracer.spatial import (
    annotate_unassigned_components_fast,
    enforce_spatial_coherence_fast,
    pre_stage2_rescue,
    reassign_unassigned_grid_pool,
    demote_small_entities,
)
from tracer.stitching import apply_stitching_to_transcripts_memory_efficient
from tracer.graph import build_grid_graph_xy, build_grid_graph_xyz

### Project folder convention

The package expects a project folder laid out like:

```
<project>/
  data/
    <name>.parquet         # input transcript table
    npmi_bs*.csv           # bootstrap PMI cache (optional)
```

`tracer.data.discover_data_files(project_dir)` locates both files. `tracer.data.load_full_df(project_dir=...)` returns the transcript DataFrame.

In [ ]:
PROJECT_DIR = Path("/Users/adeshpa6/1_Projects/01.10_Lab/GENESIS/tutorials/lung_cancer")

parquet_path, npmi_cache_path = discover_data_files(PROJECT_DIR)
print(f"data parquet:  {parquet_path}")
print(f"NPMI cache:    {npmi_cache_path or '(none — will compute)'}")

df = load_full_df(project_dir=PROJECT_DIR)
print(f"\nLoaded {len(df):,} transcripts; columns: {list(df.columns)[:8]}…")
print(f"Input cells: {df['cell_id'].astype(str).nunique():,} unique cell_ids")
print(f"Unassigned tx in input: {(df['cell_id'].astype(str) == '-1').sum():,}")

### Helpers: per-stage progression tracking

After each pipeline stage we record the number of entities and transcripts in each category. At the bottom of the notebook we'll print the full progression as a single table.

In [ ]:
def classify(label: str) -> str:
    """Bucket an entity label into cell / partial / component / unassigned."""
    s = str(label)
    if s in ("DROP", "-1", "nan", "UNASSIGNED"):
        return "unassigned"
    if s.startswith("UNASSIGNED_"):
        return "component"
    if "-" in s:
        return "partial"
    return "cell"


def state(df: pd.DataFrame, col: str) -> dict:
    """Counts at a pipeline stage. Returns dict with entity counts and tx counts per category."""
    s = df[col].astype(str)
    types = s.map(classify)
    n_ent = s.groupby(types).nunique().to_dict()
    n_tx = types.value_counts().to_dict()
    return {
        "n_cells": int(n_ent.get("cell", 0)),
        "n_partials": int(n_ent.get("partial", 0)),
        "n_components": int(n_ent.get("component", 0)),
        "tx_cells": int(n_tx.get("cell", 0)),
        "tx_partials": int(n_tx.get("partial", 0)),
        "tx_components": int(n_tx.get("component", 0)),
        "tx_unassigned": int(n_tx.get("unassigned", 0)),
    }


def fmt(s: dict) -> str:
    return (
        f"cells={s['n_cells']:,}/{s['tx_cells']:,}tx, "
        f"partials={s['n_partials']:,}/{s['tx_partials']:,}tx, "
        f"components={s['n_components']:,}/{s['tx_components']:,}tx, "
        f"unassigned={s['tx_unassigned']:,}tx"
    )


stages_log: list[tuple[str, dict]] = []

# Step 0: input state for reference (treat input cell_id as the starting partition).
_input_state = state(df.assign(_lbl=df["cell_id"].astype(str)), "_lbl")
stages_log.append(("input (cell_id)", _input_state))
print(f"Input state: {fmt(_input_state)}")

## Step 0 — Bootstrap NPMI panel

TRACER's gene-coherence checks rely on a panel of pairwise PMI values across all genes in the dataset. We compute it once with a tiered-evidence bootstrap that handles missing-pair sentinels and shared NPMI/PMI metric output, then cache to disk so subsequent runs skip the recompute.

We use **PMI** (`metric="pmi"`) rather than NPMI throughout this workflow — its unbounded scale handles strong co-association more cleanly, and the prune / rescue thresholds map naturally:

- prune positive-evidence threshold: `ln(1.5) ≈ +0.405` (50 % above independence)
- rescue negative threshold: `ln(1/3) ≈ −1.099` (3× below independence; lenient veto)

In [ ]:
PMI_THR = math.log(1.5)             # +0.405
RESCUE_NEG_THR = math.log(1.0 / 3)   # −1.099
ANNOTATE_NEG_THR = -0.1 * (PMI_THR / 0.05)  # PMI-scaled −0.1 NPMI ≡ −0.811 PMI

if npmi_cache_path is not None:
    npmi_panel = pd.read_csv(npmi_cache_path)
    print(f"Loaded cached panel from {npmi_cache_path.name}: {len(npmi_panel):,} explicit pairs")
else:
    print("Computing bootstrap NPMI panel (one-time)...")
    t0 = time.time()
    bs = compute_npmi_bootstrap(
        df, group_key="cell_id", feature_col="feature_name",
        metric="pmi", min_expected_cooccur_for_evidence=10.0,
        seed=0, show_progress=False,
    )
    import scipy.sparse as sp
    W = bs.W_sparse if sp.isspmatrix_coo(bs.W_sparse) else bs.W_sparse.tocoo()
    genes = np.asarray(bs.genes, dtype=str)
    npmi_panel = pd.DataFrame({
        "gene_i": genes[W.row],
        "gene_j": genes[W.col],
        "NPMI":   np.asarray(W.data, dtype=np.float64),
    })
    cache_out = PROJECT_DIR / "data" / "npmi_bs_full_pmi.csv"
    npmi_panel.to_csv(cache_out, index=False)
    print(f"  Computed in {time.time()-t0:.1f}s; saved to {cache_out}")

print(f"  W value range: [{npmi_panel['NPMI'].min():.3f}, {npmi_panel['NPMI'].max():.3f}]")

## Step 1 — Prune

For each input cell, drop transcripts whose gene is gene-incoherent with the rest of the cell's gene panel. The PMI threshold (`ln(1.5)`) defines what "coherent" means: a gene pair `(g, g')` is coherent if `PMI(g, g') > ln(1.5)` (≥ 50 % above independence).

Two passes:
- **Pass 1**: identify the largest gene-coherent core and split the rest into a `"<cid>-1"` partial.
- **Pass 2**: demote any partial that doesn't itself form a coherent core to `"-1"` (unassigned).

Output column: `tracer_id` (the canonical pipeline column from here on).

In [ ]:
t0 = time.time()
df_pruned, aux = prune_transcripts_fast(
    df, npmi_panel,
    cell_id_col="cell_id", gene_col="feature_name",
    threshold=PMI_THR,
    unassigned_id="-1",
    n_jobs=-1, show_progress=False,
)
s = state(df_pruned, "tracer_id")
stages_log.append(("after Prune", s))
print(f"Prune ({time.time()-t0:.1f}s): {fmt(s)}")

## Step 2 — Split

Some input cells are spatially disconnected ("Mickey-Mouse" cells from over-segmentation, or cells split across z-stacks). For each label remaining after Prune, build a 3D bin-neighborhood graph and check whether all member transcripts form a single connected component. If not, split the disconnected pieces into separate `"<cid>-N"` fragments.

**Graph primitive**: bin transcripts into 2 µm × 2 µm × 2 µm voxels; connect transcripts whose voxels are 8-adjacent in xy and within ±1 step in z. No Euclidean post-filter — pure bin connectivity.

Why this is safer than kNN: in dense tissue, a transcript's 5 nearest neighbors are mostly in *other* cells, so kNN-based connectivity artificially fragments compact cells. Bin-neighborhood guarantees that any same-label transcripts in spatially-adjacent bins are candidate edges.

In [ ]:
def grid_3d_graph(df_in, *, k=None, dist_threshold=None, coord_cols=("x", "y", "z")):
    return build_grid_graph_xyz(
        df_in, k=k, dist_threshold=dist_threshold, coord_cols=coord_cols,
        G_xy=2.0, G_z=2.0,
        xy_neighborhood="8",
        z_neighborhood_depth=1,
        exact_distance_filter=False,
    )

t0 = time.time()
df_pruned["pre_split_label"] = df_pruned["tracer_id"].astype(str)
df_pruned = enforce_spatial_coherence_fast(
    df_stitched=df_pruned,
    build_graph_fn=grid_3d_graph,
    entity_col="tracer_id",
    coord_cols=("x", "y", "z"),
    k=5, dist_threshold=5.0,    # ignored under grid_3d (kept for signature compat)
    out_col="tracer_id",
    show_progress=False,
)
df_pruned["post_split_label"] = df_pruned["tracer_id"].astype(str)
n_changed = (df_pruned["pre_split_label"] != df_pruned["post_split_label"]).sum()
s = state(df_pruned, "tracer_id")
stages_log.append(("after Split", s))
print(f"Split ({time.time()-t0:.1f}s): {fmt(s)}")
print(f"  tx whose label changed at split: {n_changed:,}")

## Step 3 — Initial Rescue

Many transcripts that Prune dropped to `"-1"` are biologically real members of nearby cells — they were just gene-coherence outliers under their original cell's panel. The Initial Rescue pass tries to re-absorb those into compatible neighbors using a **distance-priority + mean-PMI veto** rule:

1. **Spatial gating**: candidate entities are those whose member tx are in the 9-cell xy Moore neighborhood of the candidate tx at G=2 µm, within `|Δz| ≤ G·√2 ≈ 2.83 µm`.
2. **Mean-PMI veto**: an entity is rejected if the mean PMI between the candidate gene and the entity's gene panel is `≤ 0`.
3. **Distance ranking**: pick the surviving entity with the smallest min-tx distance.

**Cluster guard**: before running the rescue, find unassigned tx that have ≥3 same-bin same-gene peers (gene-coherent unassigned clusters that should later become their own UNASSIGNED_* component). Shield those from absorption.

In [ ]:
t0 = time.time()
df_pruned, n_rescued, n_skipped, rescue_stats = pre_stage2_rescue(
    df_pruned, aux=aux,
    entity_col="tracer_id", gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    out_col="tracer_id",
    G=2.0,
    pos_npmi_threshold=PMI_THR,
    neg_npmi_threshold=RESCUE_NEG_THR,
    cluster_guard_n=3,
    veto_mode="mean",
    mean_threshold=0.0,
    small_entity_guard_n=0,
)
s = state(df_pruned, "tracer_id")
stages_log.append(("after Initial Rescue", s))
print(f"Initial Rescue ({time.time()-t0:.1f}s): rescued={n_rescued:,}, shielded={n_skipped:,}")
print(f"  {fmt(s)}")

## Step 4 — Group

The unassigned transcripts that remain after Initial Rescue form gene-coherent clusters of their own — typically, ill-segmented cells the upstream segmenter missed entirely. Group bins these transcripts at 8 µm × 8 µm and forms one component per bin (the **"one bin = one candidate cell"** semantic).

Within each component, a greedy NPMI prune drops genes whose pairwise PMI to the rest is below the negative threshold (`-ln(1.5)·2 ≈ −0.811`). Components surviving the prune are labeled `"UNASSIGNED_<i>"`.

In [ ]:
def grid_self_graph(df_in, *, k=None, dist_threshold=None, coord_cols=("x", "y", "z")):
    return build_grid_graph_xy(
        df_in, k=k, dist_threshold=dist_threshold or 1.5, coord_cols=coord_cols,
        G=8.0, neighborhood="self", exact_distance_filter=False,
    )

t0 = time.time()
df_grouped = annotate_unassigned_components_fast(
    df_pruned=df_pruned, aux=aux,
    build_graph_fn=grid_self_graph,
    prune_fn=prune_genes_by_npmi_greedy,
    coord_cols=("x", "y", "z"),
    k=8, dist_threshold=1.5,
    min_comp_size=1,
    npmi_threshold=ANNOTATE_NEG_THR,
    entity_col="tracer_id",
    out_col="tracer_id",
    cell_id_col="cell_id", gene_col="feature_name",
    transcript_id_col="transcript_id",
    show_progress=False,
)
s = state(df_grouped, "tracer_id")
stages_log.append(("after Group", s))
print(f"Group ({time.time()-t0:.1f}s): {fmt(s)}")

## Step 5 — Stitch

Adjacent entities that share a gene-coherent gene panel can be merged into one. Stitch enumerates candidate merge pairs at G=2 µm 8-conn (each entity vs its grid neighbors), computes per-merge ΔC (count-based coherence change at the PMI threshold), and accepts merges with ΔC ≥ 0 and `simplicity-penalized` score ≥ threshold.

Stitch is symmetric across entity types — cells, partials, and components are all eligible to merge.

In [ ]:
df_grouped["post_stage4"] = df_grouped["tracer_id"]

t0 = time.time()
df_stitched, _ = apply_stitching_to_transcripts_memory_efficient(
    df_final=df_grouped, aux=aux,
    entity_col="post_stage4", gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    mode="count", threshold=PMI_THR, metric="pmi",
    penalize_simplicity=True, deltaC_min=0.0,
    dist_threshold=5.0,
    out_col="stitched",
    show_progress=False,
    candidate_source="grid", G=2.0, stitch_neighborhood="8",
)
s = state(df_stitched, "stitched")
stages_log.append(("after Stitch", s))
print(f"Stitch ({time.time()-t0:.1f}s): {fmt(s)}")

## Step 6 — Demote

Drop entities with fewer than 5 transcripts. Their transcripts go back to `"-1"` and become eligible for the Final Rescue pass. The minimum-size threshold guards against single-transcript "entities" that could otherwise survive Stitch.

In [ ]:
df_stitched["post_stitch_label"] = df_stitched["stitched"].astype(str)

t0 = time.time()
df_stitched, n_demoted = demote_small_entities(
    df_stitched, entity_col="stitched", out_col="stitched",
    min_size=5, unassigned_label="-1",
)
s = state(df_stitched, "stitched")
stages_log.append(("after Demote", s))
print(f"Demote ({time.time()-t0:.1f}s): n_demoted={n_demoted:,} tx")
print(f"  {fmt(s)}")

## Step 7 — Final Rescue

After Demote, some transcripts are unassigned only because their original entity got demoted (a small post-Stitch entity that lost transcripts to other merges). Final Rescue uses the same distance-priority + mean-PMI veto rule as Initial Rescue, but at G=2 µm and **without** the cluster-guard shield (no need to protect future Group input — there's no Group after this).

In [ ]:
t0 = time.time()
df_stitched, n_reassigned, post_rescue_stats = reassign_unassigned_grid_pool(
    df_stitched, aux=aux,
    entity_col="stitched", gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    out_col="stitched",
    G=2.0,
    pos_npmi_threshold=PMI_THR,
    neg_npmi_threshold=RESCUE_NEG_THR,
    only_partial_component=False,
    veto_mode="mean",
    mean_threshold=0.0,
    small_entity_guard_n=0,
)
s = state(df_stitched, "stitched")
stages_log.append(("after Final Rescue (FINAL)", s))
print(f"Final Rescue ({time.time()-t0:.1f}s): rescued={n_reassigned:,} tx")
print(f"  {fmt(s)}")

## Stage progression table

How counts evolve through each stage. Each row is one transformation; columns show how many entities in each category survive and how many transcripts each category holds.

In [ ]:
rows = []
for name, s in stages_log:
    n_total = s["tx_cells"] + s["tx_partials"] + s["tx_components"] + s["tx_unassigned"]
    n_assigned = s["tx_cells"] + s["tx_partials"] + s["tx_components"]
    pct = 100 * n_assigned / max(n_total, 1)
    rows.append({
        "stage": name,
        "cells": s["n_cells"],
        "partials": s["n_partials"],
        "components": s["n_components"],
        "tx_cells": s["tx_cells"],
        "tx_partials": s["tx_partials"],
        "tx_components": s["tx_components"],
        "tx_unassigned": s["tx_unassigned"],
        "pct_assigned": round(pct, 2),
    })
progression = pd.DataFrame(rows)
print(progression.to_string(index=False))

## Final partition

The output column `stitched` carries the final per-transcript label. Three label families:

- `"<cid>"`: cell — survived from the input segmentation (perhaps with some transcripts pruned/added).
- `"<cid>-N"`: partial — a Split-fragment of an input cell.
- `"UNASSIGNED_<i>"`: component — a new entity formed by Group from previously-unassigned transcripts.
- `"-1"`: still unassigned after Final Rescue (no compatible neighbor found).

In [ ]:
sizes = df_stitched.loc[df_stitched["stitched"].astype(str) != "-1", "stitched"].astype(str).value_counts()
kind_idx = sizes.index.to_series().map(classify)
print("per-entity transcript-count quartiles")
print("-" * 64)
for kind in ("cell", "partial", "component"):
    sub = sizes[kind_idx == kind]
    if len(sub) == 0:
        continue
    q = np.quantile(sub, [0.25, 0.5, 0.75, 0.9])
    print(f"  {kind:<10} n={len(sub):>5,}  Q1={int(q[0]):>4}  med={int(q[1]):>4}  "
          f"Q3={int(q[2]):>4}  p90={int(q[3]):>4}  max={int(sub.max()):>5}")

---

## Summary

The segmented workflow takes input cell_id assignments and applies seven transformations:

1. **Prune** removes gene-incoherent transcripts from each input cell (and may demote a cell into `"<cid>-1"` partials).
2. **Split** breaks Mickey-Mouse-shape cells into spatially-coherent fragments.
3. **Initial Rescue** absorbs unassigned transcripts into compatible neighbors using a mean-PMI veto.
4. **Group** clusters the remaining unassigned transcripts into bin-based candidate components.
5. **Stitch** merges gene-compatible adjacent entities of any type (cell ↔ partial ↔ component).
6. **Demote** drops sub-5-tx entities so Final Rescue can re-absorb their transcripts.
7. **Final Rescue** grows surviving entities by reabsorbing the demoted transcripts.

All gene-coherence judgments use a single PMI panel computed once at Step 0. Spatial gating throughout is grid-based (G=2 µm xy + ±1 z bin) so the same primitives also work on Visium HD data — see the companion `noseg_workflow.ipynb` for the no-segmentation case.